# 1. Problem Description

The goal of this assignment is to implement the A* search algorithm in a 6×6 grid environment containing obstacles. Each cell can either be free or blocked.

The agent can move in four directions: up, down, left, and right. Each move has a cost of 1.

The objective is to compute the optimal path from a start position to a fixed goal position using heuristic search.

# 2. Grid Environment

In [3]:
import heapq
import numpy as np
import pandas as pd

GRID = np.array([
[0,0,1,0,0,0],
[0,1,1,0,1,0],
[0,0,0,0,1,0],
[1,0,1,0,0,0],
[0,0,1,1,1,0],
[0,0,0,0,0,0]
])

GOAL = (5,5)

ROWS,COLS = GRID.shape

# 3. Heuristic Design

Two heuristic functions are used in this assignment to guide the A* search algorithm.

Heuristics help estimate the remaining cost from the current node to the goal node.

## Manhattan Distance

h(n) = |x_n − x_goal| + |y_n − y_goal|

This heuristic measures the horizontal and vertical distance between the current node and the goal node. It never overestimates the true path cost and therefore it is admissible and consistent.

## Chebyshev Distance

h(n) = max(|dx| , |dy|)

This heuristic takes the maximum difference between coordinates. It provides a lower bound estimate of the remaining distance and also satisfies admissibility and consistency.

# 4. Heuristic Functions

In [8]:
def heuristic_manhattan(node):

    r,c = node

    dx = abs(r - GOAL[0])
    dy = abs(c - GOAL[1])

    return dx + dy


def heuristic_chebyshev(node):

    r,c = node

    dx = abs(r - GOAL[0])
    dy = abs(c - GOAL[1])

    return max(dx,dy)

# 5. Neighbor Generation

In [10]:
def get_neighbors(node):

    r,c = node

    directions = [(1,0),(-1,0),(0,1),(0,-1)]

    neighbors = []

    for dr,dc in directions:

        nr = r + dr
        nc = c + dc

        if 0 <= nr < ROWS and 0 <= nc < COLS:

            if GRID[nr,nc] == 0:
                neighbors.append((nr,nc))

    return neighbors

# 6. Path Reconstruction

In [12]:
def reconstruct_path(parent,start,goal):

    path = []

    current = goal

    while current != start:

        path.append(current)

        current = parent[current]

    path.append(start)

    path.reverse()

    return path

# 7. A* Algorithm Implementation

In [14]:
def astar(start,heuristic):

    open_list = []

    g_cost = {start:0}

    parent = {}

    closed = set()

    h_start = heuristic(start)

    heapq.heappush(open_list,(h_start,h_start,start[0],start[1]))

    expanded_nodes = 0

    while open_list:

        f,h,r,c = heapq.heappop(open_list)

        node = (r,c)

        if node in closed:
            continue

        closed.add(node)

        expanded_nodes += 1

        if node == GOAL:

            path = reconstruct_path(parent,start,GOAL)

            return path,g_cost[node],expanded_nodes

        for neighbor in get_neighbors(node):

            if neighbor in closed:
                continue

            tentative_g = g_cost[node] + 1

            if neighbor not in g_cost or tentative_g < g_cost[neighbor]:

                g_cost[neighbor] = tentative_g

                parent[neighbor] = node

                h_val = heuristic(neighbor)

                f_val = tentative_g + h_val

                heapq.heappush(
                    open_list,
                    (f_val,h_val,neighbor[0],neighbor[1])
                )

    return None,None,expanded_nodes

# 8. Experimental Evaluation

In [16]:
start_positions = [
(0,1),
(2,0),
(4,1)
]

results = []

for start in start_positions:

    path,cost,expanded = astar(start,heuristic_manhattan)

    results.append({
        "heuristic":"Manhattan",
        "start":start,
        "path":path,
        "cost":cost,
        "expanded":expanded
    })

    path,cost,expanded = astar(start,heuristic_chebyshev)

    results.append({
        "heuristic":"Chebyshev",
        "start":start,
        "path":path,
        "cost":cost,
        "expanded":expanded
    })

# 9. Results

In [18]:
df = pd.DataFrame(results)

print(df)

for result in results:

    print("\nHeuristic:",result["heuristic"])
    print("Start:",result["start"])
    print("Path:",result["path"])
    print("Cost:",result["cost"])
    print("Expanded nodes:",result["expanded"])

   heuristic   start                                               path  cost  \
0  Manhattan  (0, 1)  [(0, 1), (0, 0), (1, 0), (2, 0), (2, 1), (2, 2...    11   
1  Chebyshev  (0, 1)  [(0, 1), (0, 0), (1, 0), (2, 0), (2, 1), (2, 2...    11   
2  Manhattan  (2, 0)  [(2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (3, 4...     8   
3  Chebyshev  (2, 0)  [(2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (3, 4...     8   
4  Manhattan  (4, 1)   [(4, 1), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)]     5   
5  Chebyshev  (4, 1)   [(4, 1), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5)]     5   

   expanded  
0        12  
1        14  
2         9  
3        13  
4         6  
5         7  

Heuristic: Manhattan
Start: (0, 1)
Path: [(0, 1), (0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (3, 4), (3, 5), (4, 5), (5, 5)]
Cost: 11
Expanded nodes: 12

Heuristic: Chebyshev
Start: (0, 1)
Path: [(0, 1), (0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (3, 3), (3, 4), (3, 5), (4, 5), (5, 5)]
Cost: 11
Expanded nodes: 14


# 10. Analysis

The experiments show that both heuristics are capable of finding the optimal path from the start position to the goal. This is expected because both heuristics are admissible and consistent, which guarantees that the A* algorithm will always produce an optimal solution.

However, the efficiency of the search process differs between the two heuristics. The Manhattan heuristic generally performs better in grid environments where movement is restricted to horizontal and vertical directions. Since the agent can only move in four directions, the Manhattan distance provides a more accurate estimate of the remaining distance to the goal.

On the other hand, the Chebyshev heuristic tends to produce smaller estimates in this environment. Because of this, the algorithm behaves closer to Uniform Cost Search and explores more nodes before reaching the goal.

From the results, it can be observed that the number of expanded nodes is usually higher when using the Chebyshev heuristic. This means that although both heuristics find the same optimal path, the Manhattan heuristic typically leads to a more efficient search.

These results highlight the importance of choosing an appropriate heuristic function when designing informed search algorithms. A heuristic that closely reflects the actual movement constraints of the environment can significantly reduce the number of explored states and improve the overall performance of the algorithm.

# 11. Conclusion

In this assignment, the A* search algorithm was implemented to solve a pathfinding problem in a grid environment with obstacles. Two heuristic functions were evaluated: Manhattan distance and Chebyshev distance.

The experimental results show that both heuristics successfully find optimal solutions. However, the Manhattan heuristic generally expands fewer nodes because it better reflects the movement constraints of the grid.

This experiment demonstrates how the choice of heuristic directly affects the efficiency of informed search algorithms such as A*.